# Training Experiment

Notebook para ejecutar un experimento de entrenamiento con la configuracion actual del proyecto y el modelo definido en `configs/config.yaml`.

In [1]:
from pathlib import Path
import os
import sys

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

project_root

WindowsPath('c:/Users/USER/Python/ProyectoCNN_week10/project_CNN_CIFAR10')

In [3]:
from datetime import datetime

import matplotlib.pyplot as plt
import torch
from torch import nn, optim

from src.data.dataset import get_cifar10_dataloaders
from src.models.cnn import build_model
from src.train.eval import evaluate
from src.train.train import (
    append_training_log,
    append_training_summary,
    ensure_outputs,
    load_config,
    train_one_epoch,
)


In [4]:
cfg = load_config('configs/config.yaml')

experiment = {
    'run_name': 'notebook_experiment',
    'epochs': cfg['training']['epochs'],
    'batch_size': cfg['training']['batch_size'],
    'learning_rate': cfg['optimizer']['lr'],
}

cfg['training']['epochs'] = experiment['epochs']
cfg['training']['batch_size'] = experiment['batch_size']
cfg['optimizer']['lr'] = experiment['learning_rate']

cfg

{'dataset': {'root': './data',
  'mean': [0.4914, 0.4822, 0.4465],
  'std': [0.247, 0.243, 0.261]},
 'model': {'name': 'resnet_lite', 'num_classes': 10},
 'training': {'device': 'cuda',
  'batch_size': 128,
  'epochs': 40,
  'num_workers': 2},
 'optimizer': {'lr': 0.0007, 'weight_decay': 0.0001},
 'paths': {'models_dir': 'outputs/models', 'logs_dir': 'outputs/logs'}}

In [5]:
device = torch.device(cfg['training']['device'] if torch.cuda.is_available() else 'cpu')
model_name = cfg['model']['name']
total_epochs = cfg['training']['epochs']

train_loader, test_loader = get_cifar10_dataloaders(
    data_dir=cfg['dataset']['root'],
    batch_size=cfg['training']['batch_size'],
    num_workers=cfg['training']['num_workers'],
    mean=tuple(cfg['dataset']['mean']),
    std=tuple(cfg['dataset']['std']),
)

model = build_model(
    model_name=model_name,
    num_classes=cfg['model']['num_classes'],
).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.02)
optimizer = optim.Adam(
    model.parameters(),
    lr=cfg['optimizer']['lr'],
    weight_decay=cfg['optimizer']['weight_decay'],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)

models_dir, logs_dir = ensure_outputs(cfg['paths'])
run_id = f"{experiment['run_name']}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
log_path = logs_dir / 'train_log.csv'
summary_path = logs_dir / 'summary.csv'
best_model_path = models_dir / f'{run_id}_best_model.pth'
last_model_path = models_dir / f'{run_id}_last_model.pth'

print(f'Run: {run_id}')
print(f'Model: {model_name}')
print(f'Device: {device}')

100.0%
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Run: notebook_experiment_20260328_203311
Model: resnet_lite
Device: cpu


In [6]:
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
}

best_accuracy = 0.0
best_epoch = 0
final_train_metrics = None
final_val_metrics = None

for epoch in range(1, total_epochs + 1):
    train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_metrics = evaluate(model, test_loader, device, criterion)
    scheduler.step()

    final_train_metrics = train_metrics
    final_val_metrics = val_metrics

    history['train_loss'].append(train_metrics['loss'])
    history['train_acc'].append(train_metrics['accuracy'])
    history['val_loss'].append(val_metrics['loss'])
    history['val_acc'].append(val_metrics['accuracy'])

    append_training_log(log_path, run_id, epoch, total_epochs, train_metrics, val_metrics)

    print(
        f"Epoch {epoch}/{total_epochs} | "
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Train Acc: {train_metrics['accuracy']:.2f}% | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.2f}%"
    )

    if val_metrics['accuracy'] > best_accuracy:
        best_accuracy = val_metrics['accuracy']
        best_epoch = epoch
        torch.save(
            {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'epoch': epoch,
                'val_accuracy': best_accuracy,
                'config': cfg,
            },
            best_model_path,
        )

append_training_summary(
    summary_path,
    run_id,
    model_name,
    total_epochs,
    cfg['optimizer']['lr'],
    cfg['training']['batch_size'],
    cfg['optimizer']['weight_decay'],
    any(isinstance(module, nn.BatchNorm2d) for module in model.modules()),
    getattr(getattr(model, 'dropout', None), 'p', 0.0),
    best_accuracy,
    best_epoch,
    final_train_metrics,
    final_val_metrics,
)

torch.save(model.state_dict(), last_model_path)

print(f'Best validation accuracy: {best_accuracy:.2f}%')
print(f'Best model saved at: {best_model_path}')
print(f'Last model saved at: {last_model_path}')

Epoch 1/40 | Train Loss: 1.4999 | Train Acc: 46.29% | Val Loss: 1.9865 | Val Acc: 42.77%
Epoch 2/40 | Train Loss: 1.0952 | Train Acc: 63.36% | Val Loss: 1.0979 | Val Acc: 64.37%
Epoch 3/40 | Train Loss: 0.9268 | Train Acc: 70.57% | Val Loss: 0.9745 | Val Acc: 70.39%
Epoch 4/40 | Train Loss: 0.8299 | Train Acc: 74.24% | Val Loss: 0.8071 | Val Acc: 75.92%
Epoch 5/40 | Train Loss: 0.7656 | Train Acc: 76.80% | Val Loss: 0.8375 | Val Acc: 76.20%
Epoch 6/40 | Train Loss: 0.7198 | Train Acc: 78.54% | Val Loss: 0.7740 | Val Acc: 77.76%
Epoch 7/40 | Train Loss: 0.6841 | Train Acc: 79.97% | Val Loss: 0.7528 | Val Acc: 77.99%
Epoch 8/40 | Train Loss: 0.6508 | Train Acc: 81.20% | Val Loss: 0.7600 | Val Acc: 78.43%
Epoch 9/40 | Train Loss: 0.6215 | Train Acc: 82.17% | Val Loss: 0.5952 | Val Acc: 83.60%
Epoch 10/40 | Train Loss: 0.5994 | Train Acc: 83.05% | Val Loss: 0.6314 | Val Acc: 82.47%
Epoch 11/40 | Train Loss: 0.5752 | Train Acc: 84.06% | Val Loss: 0.5886 | Val Acc: 84.37%
Epoch 12/40 | Train

KeyboardInterrupt: 

In [ ]:
epochs = range(1, total_epochs + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history['train_loss'], label='Train loss')
plt.plot(epochs, history['val_loss'], label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss curves')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history['train_acc'], label='Train acc')
plt.plot(epochs, history['val_acc'], label='Val acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy curves')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
{
    'run_id': run_id,
    'model_name': model_name,
    'best_val_acc': round(best_accuracy, 2),
    'best_epoch': best_epoch,
    'device': str(device),
}